# Single Agent Pipeline Project

For this assignment I had to build a simple single-agent assistant that:
- reads a user query
- figures out what the query is asking for
- routes it to the right tool
- returns the answer as a JSON dict

The agent needs to handle 3 cases:
- Math queries -> Calculator tool
- Keyword extraction queries -> Keyword tool
- Anything else -> a general fallback response

What I implemented below:
- the two tools (calculator, keyword extractor)
- the agent function that routes between them based on keywords in the query
- basic error handling so a bad query doesn't crash the agent


In [1]:
# Calculator tool

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# Keyword extractor tool

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## Agent Logic

Routing rules I used:
- if the query has the word "calculate" -> send it to the calculator
- if the query has the word "keywords" -> send it to the keyword extractor
- otherwise -> just return a general response, no tool needed


In [3]:
# Agent function

import re

def agent(query: str):
    try:
        query_lower = query.lower()

        # calculate route
        if "calculate" in query_lower:
            # pull the actual expression out of the query text
            expression = re.sub(r"[^0-9\.\+\-\*\/\(\)\s]", "", query_lower.replace("calculate", ""))
            expression = expression.strip()

            if not expression:
                return {
                    "type": "error",
                    "result": "No valid mathematical expression found in query."
                }

            calc_result = calculator(expression)
            if calc_result == "Error in calculation":
                return {
                    "type": "error",
                    "result": calc_result
                }

            return {
                "type": "calculation",
                "result": calc_result
            }

        # keywords route
        elif "keywords" in query_lower:
            # strip out the instruction phrase, keep only the actual text
            text = re.sub(r"extract keywords( from)?", "", query, flags=re.IGNORECASE).strip()

            if not text:
                return {
                    "type": "error",
                    "result": "No text found to extract keywords from."
                }

            keywords = extract_keywords(text)
            return {
                "type": "keywords",
                "result": keywords
            }

        # general fallback, no tool needed
        else:
            return {
                "type": "general",
                "result": f"I received your query: '{query}'. This looks like a general question, so no tool was used."
            }

    except Exception as e:
        return {
            "type": "error",
            "result": f"Agent failed to process query: {str(e)}"
        }

## Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```


In [4]:
# Test cases

import json

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Calculate abc",
]

for q in queries:
    print("Query:", q)
    print(json.dumps(agent(q), indent=4))
    print("-" * 50)

Query: Calculate 20 + 5
{
    "type": "calculation",
    "result": "25"
}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
{
    "type": "keywords",
    "result": [
        "intelligence",
        "transforming",
        "industries",
        "artificial"
    ]
}
--------------------------------------------------
Query: What is machine learning?
{
    "type": "general",
    "result": "I received your query: 'What is machine learning?'. This looks like a general question, so no tool was used."
}
--------------------------------------------------
Query: Calculate abc
{
    "type": "error",
    "result": "No valid mathematical expression found in query."
}
--------------------------------------------------


In [5]:
# Interactive mode - type 'exit' to stop

import json

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        print("Exiting agent.")
        break
    print(json.dumps(agent(user_input), indent=4))

Enter query (type 'exit' to stop): calculate 8+3
{
    "type": "calculation",
    "result": "11"
}
Enter query (type 'exit' to stop): exit
Exiting agent.
